# Лабораторная работа 9. Метод главных компонент, SVD и многомерное шкалирование

**Курс «Машинное обучение», 4 курс, каф. ФН1**

| | |
|---|---|
| Место в курсе | после лекции 8, завершает курс |
| Опора на лекции | лекция 8: выборочная ковариационная матрица (опр. 8.1), первая главная компонента (теорема 8.2), полная дисперсия (утв. 8.5), доля объяснённой дисперсии, численный пример 8.7, сингулярное разложение (теорема 8.9) и его связь с PCA, многомерное шкалирование; лекции 2 и 4: обусловленность, скользящий контроль |
| Трудоёмкость | 2 ч аудиторно (части 1–3) + 6 ч самостоятельно |

## Цель работы

Реализовать PCA двумя способами — через собственные векторы ковариационной матрицы и через SVD — и убедиться, что это одно и то же; проверить утверждения лекции 8 численно на примере из конспекта; применить PCA к сжатию изображений и к предобработке перед классификатором; понять, почему масштабирование меняет результат PCA; построить карту объектов по одним лишь попарным расстояниям (MDS). Последняя часть — сводная: полный цикл работы с данными на своём варианте.

## Что нужно сдать

Заполненный ноутбук `lab09_student.ipynb`, в котором:

1. выполнены все задания (ячейки с `# TODO`), код запускается сверху вниз без ошибок;
2. под каждым заданием заполнены ячейки **Вывод** — своими словами, не пересказ кода;
3. в конце — раздел «Итоги работы» с ответами на контрольные вопросы;
4. все графики подписаны (заголовок, оси, легенда).

> **Индивидуальный вариант.** Датасет и набор методов выдаются по вашему ФИО
> (см. ячейку ниже). Отчёт с чужим вариантом не принимается.

In [ ]:
# Служебная ячейка: импорты, стиль графиков, воспроизводимость.
import sys, pathlib, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

# Модули практикума (variants.py, labdata.py) ищем рядом с ноутбуком.
# Если их нет -- значит, ноутбук открыт в Colab: скачиваем из репозитория курса.
COURSE_FILES_URL = "https://raw.githubusercontent.com/sharipovaka/mltest1/main/notebooks"


def _course_modules_dir():
    here = pathlib.Path.cwd()
    for parent in [here, *here.parents][:4]:
        if (parent / "variants.py").exists():
            return str(parent)
    import urllib.request
    for name in ("variants.py", "labdata.py"):
        if not pathlib.Path(name).exists():
            urllib.request.urlretrieve(f"{COURSE_FILES_URL}/{name}", name)
            print(f"загружен {name} из репозитория курса")
    return str(here)


sys.path.insert(0, _course_modules_dir())
from variants import get_variant, describe_variant  # noqa: E402

RANDOM_STATE = 42          # единый seed на всю работу: результаты воспроизводимы
rng = np.random.default_rng(RANDOM_STATE)

plt.rcParams.update({
    "figure.figsize": (7.5, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

print("numpy", np.__version__, "| pandas", pd.__version__)

from scipy import linalg
from sklearn.datasets import load_digits, load_iris, load_wine
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.preprocessing import RobustScaler, StandardScaler
from labdata import load_personal

# SelectKBest предупреждает о константных пикселях в digits (их 3 из 64) --
# предупреждение ожидаемо и на результат не влияет.
warnings.filterwarnings("ignore", message=".*are constant.*")
warnings.filterwarnings("ignore", category=RuntimeWarning, message=".*invalid value.*")

## Индивидуальный вариант

Впишите своё ФИО (или почту) в переменную `STUDENT` — вариант вычисляется детерминированно, при повторном запуске он тот же самый.

In [ ]:
STUDENT = "Фамилия Имя Отчество"   # <-- впишите себя

variant = get_variant(STUDENT, lab=9)
describe_variant(variant)

---
# Часть 1. PCA: два вывода одного результата

Определение 8.1: выборочная ковариационная матрица центрированных данных

$$
C = \frac1\ell\sum_{i=1}^{\ell} x_i x_i^{\mathsf T} = \frac1\ell X^{\mathsf T}X .
$$

Теорема 8.2: первая главная компонента — собственный вектор $C$, отвечающий
наибольшему собственному числу, а дисперсия проекции равна этому собственному
числу. Утверждение 8.5: $\sum_j \lambda_j = \mathrm{Tr}\,C$ — полная дисперсия.

Второй путь (теорема 8.9): $X = U\Sigma V^{\mathsf T}$, и тогда
$X^{\mathsf T}X = V\Sigma^2V^{\mathsf T}$, то есть **правые сингулярные векторы
$V$ — это главные компоненты**, а $\lambda_j = \sigma_j^2/\ell$.
На практике всегда используют SVD: он не требует явно строить $C$ и не
возводит обусловленность в квадрат (работа 2, часть 4.2).

Начнём с точного воспроизведения примера 8.7 из лекции.

In [ ]:
class MyPCA:
    """PCA двумя способами: 'eig' -- через собственные векторы C, 'svd' -- через SVD.

    fit: центрировать; для 'eig' построить C = X^T X / l и взять np.linalg.eigh,
         для 'svd' -- np.linalg.svd, при этом lambda_j = sigma_j^2 / l,
         а главные компоненты -- строки Vt.
         Не забудьте отсортировать по убыванию и зафиксировать знак векторов.
    transform / inverse_transform.
    Сохраните explained_variance_, explained_variance_ratio_, total_variance_.
    """
    raise NotImplementedError


# Пример 8.7 из лекции: четыре центрированные точки
X_ex = np.array([[2.0, 1.0], [-1.0, -2.0], [1.0, 2.0], [-2.0, -1.0]])

# TODO: 1) постройте C, выведите её след и определитель;
#       2) обоими способами найдите собственные числа и первую компоненту --
#          сверьте с лекцией: lambda = 4.5 и 0.5, w_1 = (1,1)/sqrt(2), доля 0.9;
#       3) спроецируйте на первую компоненту, восстановите точки и сверьте
#          суммарную ошибку с теоретической lambda_2 * l = 2.

### Задание 1.2. Сверка со `sklearn` и цена явного построения $C$

Сравните оба своих метода со `sklearn.decomposition.PCA` на реальных данных,
а затем проверьте численное преимущество SVD: постройте плохо обусловленную
матрицу (как в работе 2) и сравните точность обоих подходов.

In [ ]:
X_w, y_w = load_wine(return_X_y=True)
X_ws = StandardScaler().fit_transform(X_w)

# TODO: 1) сверьте оба своих метода со sklearn.decomposition.PCA
#          (внимание: sklearn делит на l-1, лекция -- на l);
#       2) сравните точность 'eig' и 'svd' на матрицах с cond от 1e2 до 1e10
#          (стройте их через SVD с заданными сингулярными числами, как в работе 3;
#          матрицу центрируйте ДО построения, иначе спектр изменится).
#          Смотрите на относительную ошибку САМОГО МАЛОГО собственного числа;
#       3) постройте scree-plot и график накопленной доли дисперсии,
#          отметьте пороги 90% и 95%.
print("правило вашего варианта:", variant["n_components_rule"])

> **Вывод.** Совпали ли оба метода со `sklearn`? С какого числа обусловленности подход через $C$ начинает терять точность и почему? Сколько компонент нужно для 95 % дисперсии на `wine` и что это говорит о данных?
>
> *(ваш ответ здесь)*

---
# Часть 2. Масштабирование меняет ответ PCA

PCA ищет направления максимальной дисперсии, а дисперсия зависит от единиц
измерения. Признак «доход в рублях» имеет дисперсию в миллиарды раз большую,
чем «доля клиентов», и первая главная компонента совпадёт с ним почти точно —
независимо от содержательной важности.

Ваш вариант: `variant["scaling"]`. Сравните PCA до и после масштабирования.

In [ ]:
# TODO: сравните PCA при двух вариантах масштабирования из своего варианта:
#       для каждого нарисуйте проекцию на первые две компоненты (раскрасив
#       по классам), выведите долю дисперсии PC1 и три признака с наибольшими
#       по модулю коэффициентами в PC1. Сопоставьте с разбросом исходных признаков.
print("сравнение вашего варианта:", variant["scaling"])

> **Вывод.** Какой признак «захватил» первую компоненту без масштабирования и почему? Всегда ли нужно стандартизовать перед PCA?
>
> *(ваш ответ здесь)*

---
# Часть 3. Сжатие и восстановление

Теорема Эккарта–Янга (следствие из теоремы 8.9): усечённое SVD ранга $k$ —
**наилучшее** приближение матрицы рангом $k$ в норме Фробениуса, причём

$$
\min_{\mathrm{rank}(B) \le k}\|X - B\|_F^2 \;=\; \sum_{j > k}\sigma_j^2 ,
$$

то есть ошибка равна сумме квадратов **отброшенных** сингулярных чисел.
Проверьте это равенство численно и посмотрите на сжатие изображений.

In [ ]:
digits = load_digits()
X_dig = digits.data

# TODO: 1) проверьте теорему Эккарта--Янга: для k = 1, 5, 10, 20, 40 сравните
#          ||X - X_k||_F^2 с суммой квадратов отброшенных сингулярных чисел;
#       2) покажите восстановление нескольких цифр при k = 1, 2, 4, 8, 16, 32, 64
#          (в заголовке -- накопленная доля дисперсии);
#       3) нарисуйте среднее изображение и первые 19 главных компонент как
#          картинки 8x8 -- «собственные цифры»;
#       4) посчитайте, сколько компонент нужно для 90% дисперсии.

> **Вывод.** Совпала ли ошибка восстановления с суммой отброшенных $\sigma_j^2$? При каком $k$ цифры становятся узнаваемы и как это соотносится с долей объяснённой дисперсии? Что изображают первые главные компоненты?
>
> *(ваш ответ здесь)*

---
# Часть 4. PCA как предобработка

Снижение размерности перед обучением — компромисс: мы теряем часть информации,
но уменьшаем разброс (работа 5) и ускоряем обучение. Важно, что PCA
**не использует метки** — он максимизирует дисперсию, а не разделимость классов,
и главное направление вполне может оказаться бесполезным для классификации.

Проверьте: постройте зависимость качества от числа компонент и сравните
с обучением на всех признаках. Обязательно — с PCA **внутри** `Pipeline`
(работа 5, часть 4: иначе утечка).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.feature_selection import SelectKBest, f_classif

X_d, y_d = load_digits(return_X_y=True)

# TODO: 1) для k in [2, 5, 10, 20, 30, 40, 64] оцените скользящим контролем
#          качество Pipeline(StandardScaler -> PCA(k) -> классификатор)
#          для логистической регрессии и SVM с RBF;
#       2) сравните с обучением без PCA;
#       3) сравните PCA (не использует метки) с SelectKBest (использует метки)
#          при одинаковом числе признаков.
print("задача вашего варианта:", variant["pca_task"])

> **Вывод.** Сколько компонент достаточно, чтобы не потерять качество? Выиграл ли PCA у обучения на всех признаках? Кто оказался лучше при малом $k$ — PCA или отбор признаков по связи с целью, и почему?
>
> *(ваш ответ здесь)*

---
# Часть 5. Многомерное шкалирование

Постановка: известны только попарные расстояния $\rho(x_i, x_j)$, нужны
координаты $\tilde x_i \in \mathbb{R}^k$ с $\|\tilde x_i - \tilde x_j\| \approx \rho_{ij}$.

Классическое MDS (замечание в §5 лекции 8): по матрице квадратов расстояний
двойным центрированием восстанавливается матрица Грама

$$
B = -\tfrac12\,J D^{(2)} J, \qquad J = I - \tfrac1\ell \mathbf{1}\mathbf{1}^{\mathsf T},
$$

а координаты берутся из $k$ старших собственных пар $B$:
$\tilde X = U_k \Lambda_k^{1/2}$. Это то же вычисление, что и PCA, но для
матрицы $XX^{\mathsf T}$ размера $\ell\times\ell$ вместо $X^{\mathsf T}X$.

Классическая демонстрация: восстановить карту по таблице расстояний между
городами. Ваш вариант: `variant["embedding"]`.

In [ ]:
def classic_mds(D, k=2):
    """Классическое MDS: B = -1/2 J D^2 J, координаты -- из старших собственных пар."""
    raise NotImplementedError


cities = ["Калининград", "Санкт-Петербург", "Москва", "Ростов-на-Дону",
          "Казань", "Екатеринбург", "Новосибирск", "Иркутск", "Владивосток"]
D_km = np.array([
    [   0,  800, 1090, 1650, 1780, 2380, 3760, 5350, 7900],
    [ 800,    0,  635, 1520, 1330, 1780, 3110, 4700, 6400],
    [1090,  635,    0,  960,  720, 1420, 2810, 4200, 6430],
    [1650, 1520,  960,    0,  980, 1750, 3100, 4500, 6800],
    [1780, 1330,  720,  980,    0,  730, 2100, 3500, 5750],
    [2380, 1780, 1420, 1750,  730,    0, 1400, 2800, 5050],
    [3760, 3110, 2810, 3100, 2100, 1400,    0, 1450, 3700],
    [5350, 4700, 4200, 4500, 3500, 2800, 1450,    0, 2500],
    [7900, 6400, 6430, 6800, 5750, 5050, 3700, 2500,    0]], dtype=float)

# TODO: 1) реализуйте классическое MDS и восстановите карту по D_km;
#       2) посчитайте среднюю ошибку воспроизведения расстояний;
#       3) нарисуйте карту с подписями городов и диаграмму Шепарда
#          (истинное расстояние против расстояния на карте).
#       Учтите: MDS определяет конфигурацию с точностью до поворота и отражения.

### Задание 5.2. MDS против PCA и нелинейных методов

Если расстояния евклидовы, классическое MDS **эквивалентно** PCA: обе процедуры
используют одни и те же ненулевые собственные числа ($\sigma_j^2 = \ell\lambda_j$,
теорема 8.9). Проверьте это, а затем сравните с методом из вашего варианта.

In [ ]:
from sklearn.manifold import MDS, TSNE

X_i, y_i = load_iris(return_X_y=True)
X_is = StandardScaler().fit_transform(X_i)

# TODO: 1) убедитесь, что при евклидовой метрике классическое MDS и PCA дают
#          одну и ту же конфигурацию (сравнивайте матрицы попарных расстояний --
#          сами координаты совпадают лишь с точностью до поворота);
#       2) постройте вложения методом из своего варианта и сравните картинки.
print("метод вашего варианта:", variant["embedding"])

> **Вывод.** Совпали ли конфигурации PCA и классического MDS? В какой ситуации MDS применим, а PCA — нет? Чем принципиально отличается t-SNE от обоих?
>
> *(ваш ответ здесь)*

---
# Часть 6. Сводная: полный цикл на своей выборке

Заключительное задание курса. На индивидуальной выборке пройдите весь путь,
опираясь на работы 1–8:

1. предобработка (работа 1) — уже сделана в `load_personal`;
2. PCA: визуализация в 2D, выбор числа компонент правилом вашего варианта;
3. сравнение **трёх моделей из разных семейств** (линейная, метрическая или
   байесовская, ансамблевая) с подбором гиперпараметров по скользящему контролю
   (работа 5) — обязательно всё внутри `Pipeline`;
4. честная финальная оценка на контрольной выборке;
5. интерпретация: какие признаки важны и согласуется ли это со здравым смыслом.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score
from sklearn.neighbors import KNeighborsClassifier

data = load_personal(variant)

# TODO: 1) при необходимости бинаризуйте цель по медиане обучающей выборки;
#       2) PCA: визуализация в первых двух компонентах, выбор k правилом
#          из variant["n_components_rule"], график накопленной доли дисперсии;
#       3) три модели из РАЗНЫХ семейств (линейная, метрическая, ансамблевая)
#          с подбором гиперпараметров GridSearchCV; PCA -- шагом Pipeline,
#          чтобы не было утечки;
#       4) таблица: лучшие параметры, CV AUC и честный AUC на контроле;
#       5) перестановочная важность признаков лучшей модели + содержательная
#          интерпретация в выводе.

> **Вывод.** Какая модель победила и насколько уверенно? Помог ли PCA? Согласуется ли важность признаков с вашими ожиданиями от предметной области?
>
> *(ваш ответ здесь)*

## Итоги работы

Ответьте письменно на контрольные вопросы (по 2–4 предложения):

1. Первая главная компонента — собственный вектор $C$ с наибольшим собственным числом. Почему именно наибольшим? Приведите рассуждение из теоремы 8.2 своими словами.
2. Утверждение 8.5 говорит, что $\sum_j\lambda_j = \mathrm{Tr}\,C$. Как из этого следует правило выбора числа компонент по доле объяснённой дисперсии?
3. PCA не использует метки классов. Постройте пример двух классов в $\mathbb{R}^2$, для которых проекция на первую главную компоненту полностью уничтожает разделимость.
4. Почему на практике PCA всегда считают через SVD, а не через собственные векторы $C$? Свяжите ответ с работой 2.
5. У вас есть только матрица попарных несходств между 200 объектами, координат нет. Какие методы курса вы можете применить, а какие — нет?

### Домашнее задание

1. **Ядровой PCA.** Ядровой трюк из работы 4 переносится на PCA: вместо $X^{\mathsf T}X$ берётся центрированная матрица Грама $\tilde K = JKJ$, а проекции получаются из её старших собственных пар. Реализуйте kernel PCA с RBF-ядром и покажите на данных `make_circles`, что два вложенных кольца, неразделимые линейно и неразличимые обычным PCA, становятся линейно разделимыми в первых двух ядровых компонентах. Сравните со `sklearn.decomposition.KernelPCA` и исследуйте зависимость от $\gamma$.

2. **Сколько компонент на самом деле нужно.** Правило «95 % дисперсии» не связано с качеством решения задачи. Для своей выборки постройте **два** графика в одних осях: накопленную долю дисперсии и качество классификатора по скользящему контролю — оба как функции $k$. Покажите, что точки «95 % дисперсии» и «максимум качества» не совпадают, и объясните почему. Сформулируйте, в каких задачах разумно выбирать $k$ по дисперсии, а в каких — по качеству итоговой модели.